In [1]:
import pandas as pd
import joblib
import numpy as np

# ML Tools
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import make_pipeline

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
import xgboost as xgb
import lightgbm as lgb

# 1. Load Processed Data and Ablation Artifacts

In [2]:
X = pd.read_csv("../dataset/X_processed.csv")
y_raw = pd.read_csv("../dataset/y_processed.csv").squeeze() # squeeze converts df to series
ablation_groups = joblib.load("../models/ablation_groups.pkl")

# 2. Encode Target for SOTA Models (XGBoost/LightGBM require numeric targets)

In [3]:
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_raw)
joblib.dump(label_encoder, "../models/label_encoder.pkl") # Save for inverse transform later

['../models/label_encoder.pkl']

# 3. Train-Test Split (Holdout for future Noise/XAI testing)

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 4. Define Model Zoo with Conditional Pipelines

In [5]:
# - Linear/Distance models get a StandardScaler.
# - Tree models get raw data for interpretability.
models = {
    "LogisticRegression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
    "KNN": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5)),
    "SVM": make_pipeline(StandardScaler(), SVC(kernel='rbf', probability=True)),
    "DecisionTree": DecisionTreeClassifier(random_state=42),
    "RandomForest": RandomForestClassifier(n_estimators=200, random_state=42),
    "XGBoost": xgb.XGBClassifier(random_state=42, eval_metric='mlogloss'),
    "LightGBM": lgb.LGBMClassifier(random_state=42, verbose=-1)
}

# 5. Ablation Study & Rigorous Cross-Validation Loop

In [6]:
print("--- Starting Ablation Study & K-Fold CV ---")
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = []

for group_name, features in ablation_groups.items():
    print(f"\nEvaluating Feature Group: [{group_name.upper()}]")
    X_train_group = X_train[features]
    
    for model_name, model in models.items():
        # Calculate CV Accuracy
        scores = cross_val_score(model, X_train_group, y_train, cv=cv, scoring='accuracy', n_jobs=-1)
        mean_acc = np.mean(scores)
        
        cv_results.append({
            "Feature_Group": group_name,
            "Model": model_name,
            "CV_Mean_Accuracy": mean_acc
        })
        print(f"  {model_name}: {mean_acc:.4f}")

# Save CV results for the reporting notebook
pd.DataFrame(cv_results).to_csv("../reports/cv_ablation_results.csv", index=False)

--- Starting Ablation Study & K-Fold CV ---

Evaluating Feature Group: [ALL_FEATURES]
  LogisticRegression: 0.9682
  KNN: 0.9653
  SVM: 0.9790
  DecisionTree: 0.9852
  RandomForest: 0.9943
  XGBoost: 0.9915
  LightGBM: 0.9915

Evaluating Feature Group: [SOIL_ONLY]
  LogisticRegression: 0.6642
  KNN: 0.7477
  SVM: 0.7494
  DecisionTree: 0.7670
  RandomForest: 0.8097
  XGBoost: 0.8028
  LightGBM: 0.7977

Evaluating Feature Group: [CLIMATE_ONLY]
  LogisticRegression: 0.7585
  KNN: 0.8676
  SVM: 0.8500
  DecisionTree: 0.8847
  RandomForest: 0.9125
  XGBoost: 0.9062
  LightGBM: 0.8972


# 6. Final Training & Model Saving (Using "All Features" for the final pipeline)

In [7]:
print("\n--- Training Final Models on 'All Features' ---")
trained_models = {}

for name, model in models.items():
    # Fit on the full training set
    model.fit(X_train, y_train)
    trained_models[name] = model
    
    # Save model
    joblib.dump(model, f"../models/{name}.pkl")

# Save test sets for Notebook 4 (Evaluation & Noise Test) and Notebook 5 (XAI)
X_test.to_csv("../dataset/X_test.csv", index=False)
pd.Series(y_test, name='label').to_csv("../dataset/y_test.csv", index=False)

print("All models and test sets saved successfully.")


--- Training Final Models on 'All Features' ---
All models and test sets saved successfully.
